# Dataset Cleaning Strategy Notebook

This notebook turns the metadata document into a **dataset-specific cleaning workflow** for the four in-scope datasets in `data/`:

- `orders.csv`
- `inventory.csv`
- `vendors.csv`
- `financials.csv`

`logistics.csv` is intentionally excluded from this task.

The notebook does two things at the same time:

1. documents the cleaning strategy in plain language;
2. implements the strategy as runnable pandas code with comments.

Because the metadata contains a few contradictions, the code below prioritizes the **observed grain and column behavior in the actual files** while still following the documented business rules where they make sense.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

ROOT = Path.cwd().resolve()
DATA_DIR = r"D:\data science\Project intership\data"
MAX_BUSINESS_DATE = pd.Timestamp("2024-12-31")

cleaning_log = []

print(f"Working directory: {ROOT}")
print(f"Data directory: {DATA_DIR}")

Working directory: D:\data science\Project intership\Notebooks
Data directory: D:\data science\Project intership\data


In [2]:
def log_issue(dataset: str, issue: str, affected_rows: int, action: str) -> None:
    cleaning_log.append(
        {
            "dataset": dataset,
            "issue": issue,
            "affected_rows": int(affected_rows),
            "action": action,
        }
    )


def normalize_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    object_columns = df.select_dtypes(include="object").columns
    for column in object_columns:
        df[column] = df[column].map(lambda value: value.strip() if isinstance(value, str) else value)
    return df


def clip_series_iqr(series: pd.Series, floor: float | None = None) -> tuple[pd.Series, pd.Series]:
    valid = series.dropna()
    if valid.empty:
        return series, pd.Series(False, index=series.index)

    q1 = valid.quantile(0.25)
    q3 = valid.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    if floor is not None:
        lower = max(lower, floor)

    clipped = series.clip(lower=lower, upper=upper)
    flag = series.ne(clipped) & series.notna()
    return clipped, flag


def display_profile(name: str, df: pd.DataFrame) -> None:
    summary = pd.DataFrame(
        {
            "rows": [len(df)],
            "columns": [df.shape[1]],
            "duplicate_rows": [int(df.duplicated().sum())],
        },
        index=[name],
    )
    display(summary)
    display(df.isna().mean().sort_values(ascending=False).head(10).rename("null_share").to_frame())

## Load The Four Target Datasets

We load only the datasets requested for this task. `logistics.csv` is not used.

In [3]:
orders = pd.read_csv(r"D:\data science\Project intership\data\orders.csv")
inventory = pd.read_csv(r"D:\data science\Project intership\data\inventory.csv")
vendors = pd.read_csv(r"D:\data science\Project intership\data\vendors.csv")
financials = pd.read_csv(r"D:\data science\Project intership\data\financials.csv")

display_profile("orders", orders)
display_profile("inventory", inventory)
display_profile("vendors", vendors)
display_profile("financials", financials)

,rows,columns,duplicate_rows
orders,125660,28,0


,null_share
return_reason,0.97
actual_delivery_date,0.40
delivery_delay_days,0.40
warehouse_id,0.25
shipment_id,0.25
profit_margin_pct,0.05
customer_country,0.00
customer_segment,0.00
order_id,0.00
order_date,0.00


,rows,columns,duplicate_rows
inventory,56000,26,0


,null_share
vendor_lead_time_days,0.05
reorder_point,0.04
product_id,0.00
product_name,0.00
product_category,0.00
sku_code,0.00
warehouse_id,0.00
warehouse_location,0.00
inventory_id,0.00
snapshot_date,0.00


,rows,columns,duplicate_rows
vendors,22080,30,0


,null_share
financial_stability_score,0.12
quality_acceptance_rate,0.08
vris_score,0.07
vendor_country,0.00
vendor_name,0.00
vendor_record_id,0.00
vendor_tier,0.00
vendor_category,0.00
snapshot_month,0.00
on_time_delivery_rate,0.00


,rows,columns,duplicate_rows
financials,69300,28,3300


,null_share
shipment_id,0.88
vendor_id,0.80
order_id,0.50
financial_risk_score,0.08
working_capital_ratio,0.06
fiscal_week,0.00
fiscal_quarter,0.00
fiscal_year,0.00
transaction_type,0.00
revenue_usd,0.00


## Strategy Summary

The metadata gives us the expected business rules, but the actual files add useful nuance:

- `orders.csv`: the document says `order_id` is a primary key, but the grain also says **one order can have multiple line items**. In practice, most repeated `order_id` values are duplicate reposts of the same line, but a small number are legitimate multi-product orders. So we deduplicate at the **line-item level**, not by `order_id` alone.
- `inventory.csv`: the main issue is arithmetic consistency. Negative stock values make downstream fields like `available_stock` invalid, so the strategy favors **correcting impossible stock states and then recomputing derived columns**.
- `vendors.csv`: this table behaves like a monthly snapshot. Missing vendor-quality fields are best handled with **vendor-history-aware imputations** instead of global fills.
- `financials.csv`: duplicate rows appear to be exact reconciliation duplicates, and several financial fields are clearly derived. So the strategy is to **remove exact duplicates, standardize transaction sign conventions, and recompute derived metrics** where the rules are clear.

## Orders Cleaning Strategy

**Metadata-driven issues to address**

- duplicate order records;
- future-dated order timestamps;
- inconsistent status labels;
- negative order values;
- missing profit margin values;
- delivery fields that are only allowed to be null for certain statuses;
- product IDs that do not reconcile to inventory.

**Practical strategy**

1. Parse business dates and normalize text fields.
2. Normalize categorical labels like `order_status`.
3. Remove duplicate **line items** by comparing every business column except `last_modified_date`.
4. Cap future `order_date` values at the documented maximum date.
5. Recalculate `order_value_usd`, `gross_margin_usd`, `profit_margin_pct`, and `delivery_delay_days`.
6. Keep expected nulls when they are business-valid, but flag unexpected ones.
7. Keep orphaned `product_id` rows for traceability, but flag them for downstream exclusion from modeling.

In [4]:
orders_clean = orders.copy()
orders_clean = normalize_object_columns(orders_clean)

# Parse the date fields first so every downstream rule can use true datetime logic.
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"], errors="coerce")
orders_clean["promised_delivery_date"] = pd.to_datetime(orders_clean["promised_delivery_date"], errors="coerce")
orders_clean["actual_delivery_date"] = pd.to_datetime(orders_clean["actual_delivery_date"], errors="coerce")
orders_clean["last_modified_date"] = pd.to_datetime(orders_clean["last_modified_date"], errors="coerce")

# Normalize identifier columns to a consistent uppercase format.
id_columns = ["order_id", "customer_id", "product_id", "vendor_id", "warehouse_id", "shipment_id"]
for column in id_columns:
    orders_clean[column] = orders_clean[column].str.upper()

# Standardize categorical values that are known to vary by case or punctuation.
orders_clean["order_status"] = (
    orders_clean["order_status"]
    .str.upper()
    .replace(
        {
            "SHIP": "SHIPPED",
            "SHIP.": "SHIPPED",
        }
    )
)
orders_clean["order_status"] = orders_clean["order_status"].map(
    {
        "PENDING": "Pending",
        "CONFIRMED": "Confirmed",
        "PROCESSING": "Processing",
        "SHIPPED": "Shipped",
        "DELIVERED": "Delivered",
        "CANCELLED": "Cancelled",
        "RETURNED": "Returned",
    }
)
orders_clean["customer_segment"] = orders_clean["customer_segment"].str.title()
orders_clean["fulfillment_channel"] = orders_clean["fulfillment_channel"].replace({"3pl": "3PL", "3Pl": "3PL"}).str.title()
orders_clean["fulfillment_channel"] = orders_clean["fulfillment_channel"].replace({"3Pl": "3PL"})

# Deduplicate at the line-item level, not the order level.
# This preserves legitimate multi-product orders while removing reposted duplicates.
duplicate_subset = [column for column in orders_clean.columns if column != "last_modified_date"]
orders_clean["_last_modified_is_valid"] = orders_clean["last_modified_date"].le(MAX_BUSINESS_DATE)
orders_clean = orders_clean.sort_values(["_last_modified_is_valid", "last_modified_date"], ascending=[False, False])
duplicate_line_items = orders_clean.duplicated(subset=duplicate_subset, keep="first")
log_issue(
    "orders",
    "duplicate_line_items",
    duplicate_line_items.sum(),
    "Dropped reposted line items after sorting by valid last_modified_date.",
)
orders_clean = orders_clean.loc[~duplicate_line_items].copy()

# Future order dates are documented as data-entry errors.
future_order_mask = orders_clean["order_date"] > MAX_BUSINESS_DATE
orders_clean["future_order_date_flag"] = future_order_mask
orders_clean.loc[future_order_mask, "order_date"] = MAX_BUSINESS_DATE
log_issue(
    "orders",
    "future_order_date",
    future_order_mask.sum(),
    "Capped order_date at 2024-12-31 per metadata date range.",
)

# Negative order values are treated as invalid because there is no credit-note flag in the file.
negative_order_value_mask = orders_clean["order_value_usd"] < 0
orders_clean["negative_order_value_flag"] = negative_order_value_mask
orders_clean.loc[negative_order_value_mask, "order_value_usd"] = np.nan
log_issue(
    "orders",
    "negative_order_value",
    negative_order_value_mask.sum(),
    "Set invalid order_value_usd to NaN before recalculating derived measures.",
)

# Recalculate revenue and margin fields from the base drivers.
recalculated_order_value = (orders_clean["order_quantity"] * orders_clean["unit_price_usd"]).round(2)
order_value_mismatch_mask = ~np.isclose(
    orders_clean["order_value_usd"].fillna(-999999),
    recalculated_order_value.fillna(-999999),
)
orders_clean["order_value_mismatch_flag"] = order_value_mismatch_mask
orders_clean["order_value_usd"] = recalculated_order_value

orders_clean["gross_margin_usd"] = (orders_clean["order_value_usd"] - orders_clean["cost_of_goods_usd"]).round(2)
orders_clean["profit_margin_pct"] = np.where(
    orders_clean["order_value_usd"].gt(0),
    (orders_clean["gross_margin_usd"] / orders_clean["order_value_usd"] * 100).round(2),
    np.nan,
)
log_issue(
    "orders",
    "order_value_recalculated",
    order_value_mismatch_mask.sum(),
    "Recomputed order_value_usd, gross_margin_usd, and profit_margin_pct from quantity, price, and cost.",
)

# Delivery delay is a derived metric, so recomputing it is safer than trusting the raw file.
orders_clean["delivery_delay_days"] = (
    orders_clean["actual_delivery_date"] - orders_clean["promised_delivery_date"]
).dt.days

non_delivered_mask = orders_clean["order_status"] != "Delivered"
orders_clean.loc[non_delivered_mask, "actual_delivery_date"] = pd.NaT
orders_clean.loc[non_delivered_mask, "delivery_delay_days"] = np.nan

unexpected_missing_actual_delivery = orders_clean["order_status"].eq("Delivered") & orders_clean["actual_delivery_date"].isna()
orders_clean["unexpected_missing_actual_delivery_flag"] = unexpected_missing_actual_delivery
log_issue(
    "orders",
    "missing_actual_delivery_for_delivered_orders",
    unexpected_missing_actual_delivery.sum(),
    "Flagged delivered orders with missing actual_delivery_date; left as null because reliable imputation is not available.",
)

orders_clean["unexpected_return_reason_flag"] = orders_clean["order_status"].eq("Returned") & orders_clean["return_reason"].isna()
orders_clean.loc[orders_clean["order_status"] != "Returned", "return_reason"] = np.nan

orders_clean["unexpected_missing_warehouse_flag"] = (
    orders_clean["fulfillment_channel"] != "Drop-Ship"
) & orders_clean["warehouse_id"].isna()
orders_clean["unexpected_missing_shipment_flag"] = (
    ~orders_clean["order_status"].isin(["Pending", "Cancelled"])
) & orders_clean["shipment_id"].isna()

# Cross-table integrity check against inventory because product_id is a shared business key.
inventory_product_ids = (
    inventory["product_id"].dropna().astype(str).str.strip().str.upper().unique()
)
orders_clean["orphan_product_id_flag"] = ~orders_clean["product_id"].isin(inventory_product_ids)
log_issue(
    "orders",
    "orphan_product_id",
    orders_clean["orphan_product_id_flag"].sum(),
    "Flagged order lines whose product_id is not present in inventory.csv.",
)

orders_clean = orders_clean.drop(columns=["_last_modified_is_valid"])

orders_clean.head()

,order_id,order_date,customer_id,customer_name,customer_country,customer_segment,product_id,product_name,product_category,vendor_id,order_quantity,unit_price_usd,order_value_usd,discount_pct,cost_of_goods_usd,gross_margin_usd,profit_margin_pct,order_status,promised_delivery_date,actual_delivery_date,delivery_delay_days,fulfillment_channel,warehouse_id,shipment_id,return_reason,region,created_by,last_modified_date,future_order_date_flag,negative_order_value_flag,order_value_mismatch_flag,unexpected_missing_actual_delivery_flag,unexpected_return_reason_flag,unexpected_missing_warehouse_flag,unexpected_missing_shipment_flag,orphan_product_id_flag
1738,ORD-00103849,2024-12-08,CUST-000952,Johnson-Davis,Korea,Government,PRD-001765,Precision Component 4033,Precision,VND-000512,65,163.13,"10,603.45",15.29,"6,258.26","4,345.19",40.98,Shipped,2025-01-11,NaT,NaN,Direct,WH-13,SHP-23427798,NaN,South America,EMP-6086,2024-12-31,False,False,True,False,False,False,False,False
3347,ORD-00055643,2024-12-24,CUST-001654,Lynch-Martinez,American Samoa,Smb,PRD-000672,Raw Materials Component 6348,Raw Materials,VND-000022,4,"2,533.71","10,134.84",9.99,"6,760.45","3,374.39",33.29,Delivered,2025-03-03,2025-03-03,0.00,Direct,WH-06,SHP-83621133,NaN,South America,EMP-5909,2024-12-31,False,False,True,False,False,False,False,False
4879,ORD-00058226,2024-12-26,CUST-002983,Gill Inc,Czech Republic,Enterprise,PRD-001981,Raw Materials Component 6108,Raw Materials,VND-000291,14,32.12,449.68,16.37,231.35,218.33,48.55,Delivered,2025-01-08,2025-01-07,-1.00,Drop-Ship,NaN,SHP-67697088,NaN,APAC,EMP-4162,2024-12-31,False,False,True,False,False,False,False,False
6384,ORD-00063519,2024-12-18,CUST-001348,Rojas-Wright,Mongolia,Government,PRD-002632,Electronics Component 5078,Electronics,VND-001139,18,"5,024.31","90,437.58",3.08,"46,054.72","44,382.86",49.08,Processing,2024-12-21,NaT,NaN,3PL,WH-01,NaN,NaN,APAC,EMP-7528,2024-12-31,False,False,True,False,False,False,True,False
8916,ORD-00021334,2024-12-12,CUST-002743,"Lee, Perez and Bennett",Malawi,Government,PRD-001548,Precision Component 7221,Precision,VND-000026,55,271.18,"14,914.90",8.11,"9,203.63","5,711.27",38.29,Delivered,2025-02-23,2025-02-12,-11.00,3PL,WH-05,SHP-46271434,NaN,North America,EMP-3055,2024-12-31,False,False,True,False,False,False,False,False


## Inventory Cleaning Strategy

**Metadata-driven issues to address**

- negative `stock_on_hand` values;
- missing `reorder_point`;
- missing `vendor_lead_time_days`;
- inconsistent `available_stock`;
- outliers in `holding_cost_rate_pct`.

**Practical strategy**

1. Parse snapshot and movement dates.
2. Treat negative stock as impossible and reset it to zero while preserving a flag.
3. Force `units_reserved <= stock_on_hand`.
4. Recompute derived fields like `available_stock`, `total_inventory_value_usd`, and `stockout_flag`.
5. Impute `reorder_point` and `vendor_lead_time_days` using business-relevant group medians instead of global averages.
6. Soft-cap holding-cost outliers with an IQR rule so extreme values do not dominate analysis.

In [5]:
inventory_clean = inventory.copy()
inventory_clean = normalize_object_columns(inventory_clean)

inventory_clean["snapshot_date"] = pd.to_datetime(inventory_clean["snapshot_date"], errors="coerce")
inventory_clean["last_receipt_date"] = pd.to_datetime(inventory_clean["last_receipt_date"], errors="coerce")
inventory_clean["last_issue_date"] = pd.to_datetime(inventory_clean["last_issue_date"], errors="coerce")

for column in ["inventory_id", "product_id", "sku_code", "warehouse_id", "vendor_id"]:
    inventory_clean[column] = inventory_clean[column].str.upper()

inventory_clean["product_category"] = inventory_clean["product_category"].str.title()
inventory_clean["stock_classification"] = inventory_clean["stock_classification"].str.title()

# Negative stock is impossible for an on-hand quantity, so we convert it to zero and keep a trace flag.
negative_stock_mask = inventory_clean["stock_on_hand"] < 0
inventory_clean["negative_stock_flag"] = negative_stock_mask
inventory_clean.loc[negative_stock_mask, "stock_on_hand"] = 0
log_issue(
    "inventory",
    "negative_stock_on_hand",
    negative_stock_mask.sum(),
    "Set negative stock_on_hand values to zero and preserved a flag for auditability.",
)

# Reserved units cannot exceed what is physically on hand.
reserved_exceeds_stock_mask = inventory_clean["units_reserved"] > inventory_clean["stock_on_hand"]
inventory_clean["reserved_exceeds_stock_flag"] = reserved_exceeds_stock_mask
inventory_clean.loc[reserved_exceeds_stock_mask, "units_reserved"] = inventory_clean.loc[
    reserved_exceeds_stock_mask, "stock_on_hand"
]
log_issue(
    "inventory",
    "units_reserved_exceeds_stock",
    reserved_exceeds_stock_mask.sum(),
    "Capped units_reserved at stock_on_hand before recomputing available_stock.",
)

inventory_clean["available_stock"] = inventory_clean["stock_on_hand"] - inventory_clean["units_reserved"]
inventory_clean["stockout_flag"] = inventory_clean["available_stock"].le(0).astype(int)

# Reorder point is filled hierarchically so the imputation still respects warehouse and category context.
reorder_point_missing_mask = inventory_clean["reorder_point"].isna()
inventory_clean["reorder_point"] = inventory_clean["reorder_point"].fillna(
    inventory_clean.groupby(["product_category", "warehouse_id"])["reorder_point"].transform("median")
)
inventory_clean["reorder_point"] = inventory_clean["reorder_point"].fillna(
    inventory_clean["safety_stock_units"].astype(float)
)
log_issue(
    "inventory",
    "missing_reorder_point",
    reorder_point_missing_mask.sum(),
    "Filled reorder_point with category/warehouse medians, then safety_stock_units as a fallback.",
)

lead_time_missing_mask = inventory_clean["vendor_lead_time_days"].isna()
inventory_clean["vendor_lead_time_days"] = inventory_clean["vendor_lead_time_days"].fillna(
    inventory_clean.groupby("vendor_id")["vendor_lead_time_days"].transform("median")
)
inventory_clean["vendor_lead_time_days"] = inventory_clean["vendor_lead_time_days"].fillna(
    inventory_clean.groupby("product_category")["vendor_lead_time_days"].transform("median")
)
log_issue(
    "inventory",
    "missing_vendor_lead_time_days",
    lead_time_missing_mask.sum(),
    "Filled vendor lead time from vendor-level medians, then product-category medians.",
)

inventory_clean["holding_cost_rate_pct"], holding_cost_outlier_mask = clip_series_iqr(
    inventory_clean["holding_cost_rate_pct"], floor=0
)
inventory_clean["holding_cost_rate_outlier_flag"] = holding_cost_outlier_mask
log_issue(
    "inventory",
    "holding_cost_rate_outliers",
    holding_cost_outlier_mask.sum(),
    "Winsorized holding_cost_rate_pct with an IQR cap to reduce the effect of extreme values.",
)

# Derived value fields should always follow the corrected stock and unit cost values.
inventory_clean["total_inventory_value_usd"] = (
    inventory_clean["stock_on_hand"] * inventory_clean["unit_cost_usd"]
).round(2)

inventory_clean.head()

,inventory_id,snapshot_date,product_id,product_name,product_category,sku_code,warehouse_id,warehouse_location,stock_on_hand,units_reserved,available_stock,reorder_point,reorder_quantity,safety_stock_units,max_stock_level,days_of_supply,vendor_id,vendor_lead_time_days,last_receipt_date,last_issue_date,stock_age_days,stock_classification,holding_cost_rate_pct,unit_cost_usd,total_inventory_value_usd,stockout_flag,negative_stock_flag,reserved_exceeds_stock_flag,holding_cost_rate_outlier_flag
0,INV-0000000001,2023-08-06,PRD-001172,Precision Component 8522,Precision,SKU-1172-01,WH-01,Singapore,218,58,160,102.00,170,33,442,43.60,VND-000758,17.00,2023-05-20,2023-07-28,464,Cx,3.10,11.13,"2,426.34",0,False,False,False
1,INV-0000000002,2024-01-26,PRD-001526,Raw Materials Component 1668,Raw Materials,SKU-1526-10,WH-10,"Sydney, Australia",66,10,56,57.60,96,22,249,16.50,VND-000889,12.00,2023-12-08,2024-01-07,109,Az,3.87,92.73,"6,120.18",0,False,False,False
2,INV-0000000003,2023-03-04,PRD-002032,Precision Component 3157,Precision,SKU-2032-09,WH-09,"Johannesburg, South Africa",313,34,279,873.60,1456,225,3785,12.04,VND-001038,28.00,2022-12-19,2023-02-01,385,Ax,1.22,369.53,"115,662.89",0,False,False,False
3,INV-0000000004,2022-07-18,PRD-000679,Logistics Component 1641,Logistics,SKU-0679-14,WH-14,"Seoul, South Korea",76,15,61,237.60,396,76,1029,6.91,VND-000158,18.00,2022-04-25,2022-05-17,598,Cy,6.41,21.85,"1,660.60",0,False,False,True
4,INV-0000000005,2022-11-08,PRD-002651,Logistics Component 920,Logistics,SKU-2651-13,WH-13,"Cairo, Egypt",12,3,9,878.40,1464,153,3806,1.00,VND-001084,61.00,2022-10-28,2022-08-19,170,By,1.44,11.77,141.24,0,False,False,False


## Vendors Cleaning Strategy

**Metadata-driven issues to address**

- missing quality and financial stability measures;
- lead-time inconsistencies relative to contracted lead time;
- region and category normalization;
- contract dates that need lifecycle flags;
- missing `vris_score`.

**Practical strategy**

1. Treat this table as a monthly time series per vendor.
2. Parse `snapshot_month` and contract dates.
3. Impute quality performance using recent vendor history first, then vendor-category medians.
4. Impute `financial_stability_score` with vendor-history-aware fills.
5. Flag lead-time deviations from contract instead of overwriting them, because they can be real performance signals.
6. Add contract lifecycle flags.
7. Rebuild `vris_score` only when it is missing, using a transparent proxy formula based on the risk drivers available in the file.

In [7]:
vendors_clean = vendors.copy()
vendors_clean = normalize_object_columns(vendors_clean)

vendors_clean["snapshot_month"] = pd.to_datetime(vendors_clean["snapshot_month"], errors="coerce")
vendors_clean["contract_start_date"] = pd.to_datetime(vendors_clean["contract_start_date"], errors="coerce")
vendors_clean["contract_expiry_date"] = pd.to_datetime(vendors_clean["contract_expiry_date"], errors="coerce")

for column in ["vendor_record_id", "vendor_id"]:
    vendors_clean[column] = vendors_clean[column].str.upper()

vendors_clean["vendor_tier"] = vendors_clean["vendor_tier"].str.title()
vendors_clean["vendor_category"] = vendors_clean["vendor_category"].str.title()
vendors_clean["risk_category"] = vendors_clean["risk_category"].str.title()

vendors_clean = vendors_clean.sort_values(["vendor_id", "snapshot_month"]).copy()

# The table is a monthly snapshot, so vendor-history-aware imputations are more defensible than global means.
quality_missing_mask = vendors_clean["quality_acceptance_rate"].isna()
rolling_quality = vendors_clean.groupby("vendor_id")["quality_acceptance_rate"].transform(
    lambda series: series.ffill().rolling(3, min_periods=1).mean()
)
vendors_clean["quality_acceptance_rate"] = vendors_clean["quality_acceptance_rate"].fillna(rolling_quality)
vendors_clean["quality_acceptance_rate"] = vendors_clean["quality_acceptance_rate"].fillna(
    vendors_clean.groupby("vendor_category")["quality_acceptance_rate"].transform("median")
)
log_issue(
    "vendors",
    "missing_quality_acceptance_rate",
    quality_missing_mask.sum(),
    "Filled with vendor-level 3-month rolling history, then vendor-category median.",
)

financial_stability_missing_mask = vendors_clean["financial_stability_score"].isna()
vendors_clean["financial_stability_score"] = vendors_clean["financial_stability_score"].fillna(
    vendors_clean.groupby("vendor_id")["financial_stability_score"].transform("median")
)
vendors_clean["financial_stability_score"] = vendors_clean["financial_stability_score"].fillna(
    vendors_clean.groupby("vendor_category")["financial_stability_score"].transform("median")
)
log_issue(
    "vendors",
    "missing_financial_stability_score",
    financial_stability_missing_mask.sum(),
    "Filled with vendor-level medians, then vendor-category medians.",
)

lead_time_missing_mask = vendors_clean["lead_time_avg_days"].isna()
vendors_clean["lead_time_avg_days"] = vendors_clean["lead_time_avg_days"].fillna(vendors_clean["contract_lead_time_days"])
log_issue(
    "vendors",
    "missing_lead_time_avg_days",
    lead_time_missing_mask.sum(),
    "Filled missing lead_time_avg_days with contract_lead_time_days.",
)

# Out-of-band lead times are kept because they can reflect true supplier performance,
# but we still flag them because the metadata treats them as a data-quality concern.
vendors_clean["lead_time_inconsistency_flag"] = (
    vendors_clean["lead_time_avg_days"] < vendors_clean["contract_lead_time_days"] * 0.8
) | (
    vendors_clean["lead_time_avg_days"] > vendors_clean["contract_lead_time_days"] * 1.2
)
log_issue(
    "vendors",
    "lead_time_inconsistency",
    vendors_clean["lead_time_inconsistency_flag"].sum(),
    "Flagged lead_time_avg_days outside the contractual +/-20% band without overwriting the raw business signal.",
)

vendors_clean["contract_expired_flag"] = (
    vendors_clean["contract_expiry_date"].notna()
) & (
    vendors_clean["contract_expiry_date"] < vendors_clean["snapshot_month"]
)
vendors_clean["contract_renewal_due_flag"] = (
    vendors_clean["contract_expiry_date"].notna()
) & (
    (vendors_clean["contract_expiry_date"] - vendors_clean["snapshot_month"]).dt.days.between(0, 90)
)

# Concentration risk can be rebuilt from spend share if it is ever missing.
category_spend = vendors_clean.groupby(["snapshot_month", "vendor_category"])["procurement_spend_usd"].transform("sum")
recalculated_concentration = (
    vendors_clean["procurement_spend_usd"]
    .div(category_spend.where(category_spend.gt(0)))
    .mul(100)
    .round(2)
)
vendors_clean["concentration_risk_pct"] = vendors_clean["concentration_risk_pct"].fillna(recalculated_concentration)

# Rebuild VRIS only when it is missing. This keeps the original business score whenever it exists.
vris_missing_mask = vendors_clean["vris_score"].isna()
lead_time_risk = np.where(
    vendors_clean["contract_lead_time_days"].gt(0),
    (vendors_clean["lead_time_variance_days"] / vendors_clean["contract_lead_time_days"] * 100).clip(0, 100),
    0,
)
vris_proxy = (
    0.20 * (100 - vendors_clean["on_time_delivery_rate"].clip(0, 100))
    + 0.20 * (100 - vendors_clean["quality_acceptance_rate"].clip(0, 100))
    + 0.15 * (vendors_clean["defect_rate_pct"].clip(0, 15) / 15 * 100)
    + 0.15 * lead_time_risk
    + 0.10 * vendors_clean["concentration_risk_pct"].clip(0, 100)
    + 0.10 * (100 - vendors_clean["financial_stability_score"].clip(0, 100))
    + 0.10 * (vendors_clean["active_dispute_flag"].clip(0, 1) * 100)
).clip(0, 100)
vendors_clean["vris_score"] = vendors_clean["vris_score"].fillna(vris_proxy.round(2))
vendors_clean["risk_category"] = np.select(
    [
        vendors_clean["vris_score"] >= 70,
        vendors_clean["vris_score"] >= 40,
    ],
    [
        "High",
        "Medium",
    ],
    default="Low",
)
log_issue(
    "vendors",
    "missing_vris_score",
    vris_missing_mask.sum(),
    "Filled missing vris_score values with a transparent proxy based on service, quality, lead-time, and financial-risk drivers.",
)

vendors_clean.head()

,vendor_record_id,vendor_id,vendor_name,vendor_country,vendor_region,vendor_tier,vendor_category,snapshot_month,on_time_delivery_rate,quality_acceptance_rate,quality_rejection_count,defect_rate_pct,lead_time_avg_days,lead_time_variance_days,contract_lead_time_days,procurement_spend_usd,purchase_order_count,invoice_accuracy_rate,payment_terms_days,days_payable_outstanding,contract_start_date,contract_expiry_date,concentration_risk_pct,financial_stability_score,esg_compliance_score,past_disruption_count,active_dispute_flag,preferred_vendor_flag,vris_score,risk_category,lead_time_inconsistency_flag,contract_expired_flag,contract_renewal_due_flag
12576,VR-00012577,VND-000001,Mccoy Inc,Moldova,MEA,Tier-3,Services,2021-01-01,79.35,70.16,2,0.01,28.82,0.19,28,"54,139.13",12,94.32,15,26.47,2019-12-01,2023-02-22,30.98,51.69,38.94,0,0,0,67.89,Medium,False,False,False
12577,VR-00012578,VND-000001,Mccoy Inc,Moldova,MEA,Tier-3,Services,2021-02-01,77.81,66.00,3,0.42,32.69,5.92,36,"15,166.98",11,84.46,15,19.07,2019-12-01,2023-04-17,17.51,75.41,60.46,0,0,0,74.60,High,False,False,False
12578,VR-00012579,VND-000001,Mccoy Inc,Moldova,MEA,Tier-3,Services,2021-03-01,65.16,78.30,3,0.09,24.28,2.87,26,"45,117.60",9,93.69,15,2.80,2019-12-01,2023-10-01,7.44,50.84,51.19,0,0,0,70.49,High,False,False,False
12579,VR-00012580,VND-000001,Mccoy Inc,Moldova,MEA,Tier-3,Services,2021-04-01,67.24,78.30,0,0.16,14.48,1.93,20,"2,475.90",8,85.46,15,22.32,2019-12-01,2024-02-17,30.29,76.59,45.25,1,0,0,71.32,High,True,False,False
12580,VR-00012581,VND-000001,Mccoy Inc,Moldova,MEA,Tier-3,Services,2021-05-01,69.60,80.32,1,0.11,15.73,0.01,15,"19,350.06",15,86.41,15,19.09,2019-12-01,2024-02-05,43.92,77.02,40.61,0,1,0,69.98,Medium,False,False,False


## Financials Cleaning Strategy

**Metadata-driven issues to address**

- exact duplicate records from reconciliation;
- missing working-capital and financial-risk metrics;
- sign inconsistencies in `cash_flow_usd`;
- derived columns that should agree with their components.

**Practical strategy**

1. Parse the financial timeline fields.
2. Remove exact duplicates first.
3. Normalize transaction-type labels and enforce sign conventions for inflows vs outflows.
4. Recompute clear derived fields:
   - `fiscal_week`, `fiscal_quarter`, `fiscal_year`
   - `gross_profit_usd`
   - `working_capital_usd`
   - `working_capital_ratio`
   - `cash_position_usd`
5. Rebuild `financial_risk_score` only when it is missing, using a simple proxy driven by liquidity and conversion-cycle pressure.
6. Recompute `risk_flag` from the cleaned score and working-capital ratio.

In [8]:
financials_clean = financials.copy()
financials_clean = normalize_object_columns(financials_clean)

financials_clean["record_date"] = pd.to_datetime(financials_clean["record_date"], errors="coerce")
for column in ["finance_record_id", "order_id", "vendor_id", "shipment_id"]:
    financials_clean[column] = financials_clean[column].str.upper()

financials_clean["transaction_type"] = financials_clean["transaction_type"].replace(
    {
        "sla penalty": "SLA Penalty",
        "Sla Penalty": "SLA Penalty",
        "cogs": "COGS",
    }
)

exact_duplicate_mask = financials_clean.duplicated(keep="first")
log_issue(
    "financials",
    "exact_duplicate_rows",
    exact_duplicate_mask.sum(),
    "Dropped exact duplicate rows created by reconciliation duplicates.",
)
financials_clean = financials_clean.loc[~exact_duplicate_mask].copy()

# Rebuild the fiscal calendar directly from record_date so it is internally consistent.
iso_calendar = financials_clean["record_date"].dt.isocalendar()
financials_clean["fiscal_week"] = iso_calendar.week.astype("Int64")
financials_clean["fiscal_quarter"] = "Q" + financials_clean["record_date"].dt.quarter.astype("Int64").astype(str)
financials_clean["fiscal_year"] = financials_clean["record_date"].dt.year.astype("Int64")

# Revenue should be a positive inflow; expense-like rows should be negative cash flows.
inflow_types = {"Revenue"}
outflow_types = {"Procurement", "COGS", "Logistics Cost", "Operational Expense", "SLA Penalty", "Interest Expense"}

cash_flow_sign_issue_mask = (
    (financials_clean["transaction_type"].isin(inflow_types) & financials_clean["cash_flow_usd"].lt(0))
    | (financials_clean["transaction_type"].isin(outflow_types) & financials_clean["cash_flow_usd"].gt(0))
)
financials_clean["cash_flow_sign_issue_flag"] = cash_flow_sign_issue_mask
financials_clean.loc[
    financials_clean["transaction_type"].isin(inflow_types), "cash_flow_usd"
] = financials_clean.loc[
    financials_clean["transaction_type"].isin(inflow_types), "cash_flow_usd"
].abs()
financials_clean.loc[
    financials_clean["transaction_type"].isin(outflow_types), "cash_flow_usd"
] = -financials_clean.loc[
    financials_clean["transaction_type"].isin(outflow_types), "cash_flow_usd"
].abs()
log_issue(
    "financials",
    "cash_flow_sign_inconsistency",
    cash_flow_sign_issue_mask.sum(),
    "Forced Revenue rows positive and expense-like rows negative in cash_flow_usd.",
)

# Gross profit and working capital are explicit formulas in the metadata.
financials_clean["gross_profit_usd"] = (
    financials_clean["revenue_usd"] - financials_clean["procurement_cost_usd"]
).round(2)
financials_clean["working_capital_usd"] = (
    financials_clean["accounts_receivable_usd"]
    + financials_clean["inventory_value_usd"]
    - financials_clean["accounts_payable_usd"]
).round(2)

current_assets = financials_clean["accounts_receivable_usd"] + financials_clean["inventory_value_usd"]
financials_clean["working_capital_ratio"] = np.where(
    financials_clean["accounts_payable_usd"].gt(0),
    (current_assets / financials_clean["accounts_payable_usd"]).round(4),
    np.nan,
)

# Cash position is cumulative by date after the sign cleanup.
financials_clean = financials_clean.sort_values(["record_date", "finance_record_id"]).copy()
financials_clean["cash_position_usd"] = financials_clean["cash_flow_usd"].cumsum().round(2)

# Missing risk scores are filled with a transparent proxy rather than a blind mean.
def minmax_risk(series: pd.Series, invert: bool = False) -> pd.Series:
    valid = series.astype(float)
    min_value = valid.min()
    max_value = valid.max()
    if pd.isna(min_value) or pd.isna(max_value) or min_value == max_value:
        scaled = pd.Series(50.0, index=series.index)
    else:
        scaled = (valid - min_value) / (max_value - min_value) * 100
    return 100 - scaled if invert else scaled


financial_risk_missing_mask = financials_clean["financial_risk_score"].isna()
risk_proxy = (
    0.35 * minmax_risk(financials_clean["working_capital_ratio"], invert=True)
    + 0.25 * minmax_risk(financials_clean["days_sales_outstanding"])
    + 0.20 * minmax_risk(financials_clean["cash_conversion_cycle"])
    + 0.20 * minmax_risk((-financials_clean["cash_flow_usd"]).clip(lower=0))
).clip(0, 100)
financials_clean["financial_risk_score"] = financials_clean["financial_risk_score"].fillna(risk_proxy.round(2))
log_issue(
    "financials",
    "missing_financial_risk_score",
    financial_risk_missing_mask.sum(),
    "Filled missing financial_risk_score values with a liquidity- and cycle-based proxy.",
)

financials_clean["risk_flag"] = (
    (financials_clean["financial_risk_score"] > 70)
    | (financials_clean["working_capital_ratio"] < 1.0)
).astype(int)

financials_clean["procurement_gt_revenue_flag"] = (
    financials_clean["revenue_usd"].gt(0)
) & (
    financials_clean["procurement_cost_usd"] > financials_clean["revenue_usd"]
)

financials_clean.head()

,finance_record_id,record_date,fiscal_week,fiscal_quarter,fiscal_year,order_id,vendor_id,shipment_id,transaction_type,revenue_usd,procurement_cost_usd,logistics_cost_usd,operational_cost_usd,sla_penalty_usd,gross_profit_usd,ebitda_usd,accounts_receivable_usd,accounts_payable_usd,inventory_value_usd,working_capital_usd,working_capital_ratio,cash_flow_usd,cash_position_usd,days_sales_outstanding,days_payable_outstanding,cash_conversion_cycle,financial_risk_score,risk_flag,cash_flow_sign_issue_flag,procurement_gt_revenue_flag
683,FIN-0000000684,2021-01-01,53,Q1,2021,NaN,NaN,NaN,Revenue,"19,046.90",0.00,0.00,0.00,0.00,"19,046.90","19,046.90","117,194.64","96,579.58","61,951.35","82,566.41",1.85,"944,763.64","944,763.64",83.80,61.49,55.75,23.22,0,False,False
4336,FIN-0000004337,2021-01-01,53,Q1,2021,NaN,NaN,NaN,Revenue,"2,346.99",0.00,0.00,0.00,0.00,"2,346.99","2,346.99","35,115.56","79,842.20","174,552.97","129,826.33",2.63,"27,541.31","972,304.95",30.85,51.68,9.46,28.74,0,True,False
5611,FIN-0000005612,2021-01-01,53,Q1,2021,NaN,NaN,NaN,Operational Expense,0.00,0.00,0.00,"1,304.82",0.00,0.00,"-1,304.82","49,063.98","7,330.96","76,210.32","117,943.34",17.09,"-1,300,154.76","-327,849.81",71.62,56.55,55.10,23.03,0,True,False
8302,FIN-0000008303,2021-01-01,53,Q1,2021,ORD-00085325,NaN,NaN,COGS,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"69,557.03","1,559,655.32","5,247.71","-1,484,850.58",0.05,"-526,501.26","-854,351.07",48.36,84.42,6.08,42.53,1,True,False
9230,FIN-0000009231,2021-01-01,53,Q1,2021,ORD-00025553,NaN,NaN,Revenue,"7,203.28",0.00,0.00,0.00,0.00,"7,203.28","7,203.28","108,622.89","542,048.65","32,284.39","-401,141.37",0.26,"793,306.62","-61,044.45",49.01,87.99,-0.54,38.10,1,False,False


## Final Quality Summary

This summary keeps the notebook easy to audit. It shows the cleaned dataset sizes and every cleaning action that was applied.

In [9]:
cleaned_datasets = {
    "orders": orders_clean,
    "inventory": inventory_clean,
    "vendors": vendors_clean,
    "financials": financials_clean,
}

summary_rows = []
for name, frame in cleaned_datasets.items():
    summary_rows.append(
        {
            "dataset": name,
            "rows": len(frame),
            "columns": frame.shape[1],
            "remaining_nulls": int(frame.isna().sum().sum()),
        }
    )

display(pd.DataFrame(summary_rows))
display(pd.DataFrame(cleaning_log))

,dataset,rows,columns,remaining_nulls
0,orders,122731,36,278150
1,inventory,56000,29,0
2,vendors,22080,33,0
3,financials,66000,30,143797


,dataset,issue,affected_rows,action
0,orders,duplicate_line_items,2929,Dropped reposted line items after sorting by v...
1,orders,future_order_date,5020,Capped order_date at 2024-12-31 per metadata d...
2,orders,negative_order_value,2511,Set invalid order_value_usd to NaN before reca...
3,orders,order_value_recalculated,122701,"Recomputed order_value_usd, gross_margin_usd, ..."
4,orders,missing_actual_delivery_for_delivered_orders,0,Flagged delivered orders with missing actual_d...
5,orders,orphan_product_id,1257,Flagged order lines whose product_id is not pr...
6,inventory,negative_stock_on_hand,3360,Set negative stock_on_hand values to zero and ...
7,inventory,units_reserved_exceeds_stock,3177,Capped units_reserved at stock_on_hand before ...
8,inventory,missing_reorder_point,2240,Filled reorder_point with category/warehouse m...
9,inventory,missing_vendor_lead_time_days,2800,Filled vendor lead time from vendor-level medi...


## Optional Export

If you want to persist the cleaned outputs after reviewing them, uncomment and run the next cell.

In [ ]:
# OUTPUT_DIR = DATA_DIR / "cleaned"
# OUTPUT_DIR.mkdir(exist_ok=True)
#
# for dataset_name, frame in cleaned_datasets.items():
#     frame.to_csv(OUTPUT_DIR / f"{dataset_name}_clean.csv", index=False)
#
# print(f"Cleaned files written to: {OUTPUT_DIR}")